# C3-Z26 C3-ROI V2 — Pilot C full 5-fold
Mỗi runtime chạy một fold. Fold 1 đã hoàn tất; dùng FOLD=2,3,4,5 lần lượt.
Pilot C = mild artifact augmentation + consistency loss 0.30.


In [ ]:
FOLD = 2  # chỉ đổi thành 3, 4, 5 ở runtime kế tiếp
assert FOLD in {2, 3, 4, 5}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import zipfile, shutil, subprocess, sys
DATA_DIR = Path('/content/drive/MyDrive/RSNA_DATA')
MAIN_CODE_ZIP = DATA_DIR / 'C3_Z26_C3_ROI_T4_CODE_V2.zip'
DATA_ZIP = DATA_DIR / 'C3_Z26_COMBO_V2_FINAL.zip'
PILOT_ZIP = DATA_DIR / 'C3_Z26_C3_ROI_V2_PILOT_C_5FOLD_CODE_V1.zip'
for path in (MAIN_CODE_ZIP, DATA_ZIP, PILOT_ZIP):
    assert path.is_file(), f'Thiếu {path}'
if not Path('/content/C3_Z26_C3_ROI_V2/manifests/fold_2_train.csv').is_file():
    with zipfile.ZipFile(MAIN_CODE_ZIP) as zf: zf.extractall('/content')
if not Path('/content/C3_Z26_COMBO_V2').is_dir():
    with zipfile.ZipFile(DATA_ZIP) as zf: zf.extractall('/content')
shutil.rmtree('/content/p1_baseline', ignore_errors=True)
with zipfile.ZipFile(PILOT_ZIP) as zf: zf.extractall('/content')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/content/requirements_colab.txt'], check=True)
assert Path(f'/content/C3_Z26_C3_ROI_V2/manifests/fold_{FOLD}_train.csv').is_file()
assert Path('/content/C3_Z26_COMBO_V2').is_dir()
print('SETUP PASS — FOLD', FOLD)


In [ ]:
# Có thể chạy lại cell này nếu Colab ngắt; runner chỉ resume checkpoint cùng fold.
runner = Path('/content/C3_Z26_C3_ROI_V2_PILOTS/pilot_bc_runner.py')
subprocess.run([sys.executable, '-u', str(runner), '--pilot', 'C', '--fold', str(FOLD)], check=True)


In [ ]:
evaluator = Path('/content/C3_Z26_C3_ROI_V2_PILOTS/evaluate_pilot_bc.py')
subprocess.run([sys.executable, '-u', str(evaluator), '--pilot', 'C', '--fold', str(FOLD)], check=True)
run_id = f'C3_Z26_C3_ROI_V2_PILOT_C_V2_FOLD_{FOLD}_SEED_42'
run_dir = DATA_DIR / 'C3_Z26_C3_ROI_V2_PILOTS' / 'runs' / run_id
for name in ('best_mae.ckpt', 'run_state.json', 'train.log', 'artifact_robustness_predictions.csv', 'artifact_robustness_report.json'):
    assert (run_dir / name).is_file(), run_dir / name
print('FOLD SAVED:', run_dir)


## Sau khi Fold 2–5 đều có report
Chạy cell cuối trong một runtime T4 để ghép Fold 1–5 và inference baseline artifact nếu chưa cache.


In [ ]:
aggregator = Path('/content/C3_Z26_C3_ROI_V2_PILOTS/evaluate_pilot_c_5fold.py')
subprocess.run([sys.executable, '-u', str(aggregator), '--bootstrap-repetitions', '20000'], check=True)
